<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np

# 1. 구글 드라이브 연결 및 2025년 청소년건강행태조사 원시자료 로드 (10주차 코드 그대로)
drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')

# 2. 변수 생성 (행을 삭제하지 않고 전체 54,170명 유지 - 10주차 코드 그대로)
# 고등학생 도메인 (GRADE 4, 5, 6 = 고1, 고2, 고3)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)

# 아침식사 결식 변수 (F_BR: 1~3은 주 5회 이상 결식 -> 1, 4~8은 비결식 -> 0)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notna(), 0, np.nan))

# BMI 및 비만 변수 (BMI >= 25 -> 1, < 25 -> 0)
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notna(), 0, np.nan))

# 10주차 최종 유효 분석 대상자 표시 (in_analysis 그대로 유지)
df['in_analysis'] = ((df['domain_hs'] == 1) & df['bf_skip'].notna() & df['bmi'].notna()).astype(int)

print(f"전체 표본 수: {len(df):,}명")
print(f"10주차 Table 1 유효 분석 대상자 (in_analysis == 1): {df['in_analysis'].sum():,}명")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 표본 수: 54,170명
10주차 Table 1 유효 분석 대상자 (in_analysis == 1): 25,077명


/tmp/ipykernel_2286/132412296.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_2286/132412296.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
/tmp/ipykernel_2286/132412296.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. 

In [2]:
# 최소충분보정집합(교란변수): 성별(SEX), 학년(GRADE), 주관적 가정경제상태(E_SES 또는 PR_INC)
df['sex_bin'] = np.where(df['SEX'] == 1, 1, np.where(df['SEX'] == 2, 0, np.nan))
ses_col = 'E_SES' if 'E_SES' in df.columns else 'PR_INC'
df[ses_col] = df[ses_col].replace(9, np.nan)

# 보정변수가 모두 있는 완전 사례 표시변수 (in_complete)
covs = ['sex_bin', 'GRADE', ses_col]
df['in_complete'] = ((df['in_analysis'] == 1) & df[covs].notna().all(axis=1)).astype(int)

n_table1 = int(df['in_analysis'].sum())
n_complete = int(df['in_complete'].sum())
n_diff = n_table1 - n_complete

print("=== [완전 사례 표시 및 인원 차이 확인] ===")
print(f"1. 10주차 Table 1 분석 대상자 (in_analysis): {n_table1:,}명")
print(f"2. 보정변수 모두 유효한 완전 사례 (in_complete): {n_complete:,}명")
print(f"3. Table 1 분석 대상과의 인원 차이: {n_diff:,}명 제외 (보정변수 {ses_col} 결측)")
print(f"   - 결식군 (bf_skip == 1): {int(((df['in_complete'] == 1) & (df['bf_skip'] == 1)).sum()):,}명")
print(f"   - 비결식군 (bf_skip == 0): {int(((df['in_complete'] == 1) & (df['bf_skip'] == 0)).sum()):,}명")

=== [완전 사례 표시 및 인원 차이 확인] ===
1. 10주차 Table 1 분석 대상자 (in_analysis): 25,077명
2. 보정변수 모두 유효한 완전 사례 (in_complete): 25,070명
3. Table 1 분석 대상과의 인원 차이: 7명 제외 (보정변수 E_SES 결측)
   - 결식군 (bf_skip == 1): 11,375명
   - 비결식군 (bf_skip == 0): 13,695명


/tmp/ipykernel_2286/3766998386.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sex_bin'] = np.where(df['SEX'] == 1, 1, np.where(df['SEX'] == 2, 0, np.nan))
/tmp/ipykernel_2286/3766998386.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['in_complete'] = ((df['in_analysis'] == 1) & df[covs].notna().all(axis=1)).astype(int)


In [3]:
import scipy.linalg as la
from scipy.stats import norm
import numpy as np
import pandas as pd

def fit_svy_domain_logit(use_covariates, data, domain_col='in_complete', w_col='W', h_col='STRATA', c_col='CLUSTER'):
    """
    행을 한 줄도 지우지 않고 전체 54,170명을 유지한 상태에서
    domain_col == 1(같은 사람)에 대해 가중치(W), 층(STRATA), 집락(CLUSTER)을 반영한
    설계 기반 로지스틱 회귀모형을 적합한다.
    """
    d = (data[domain_col] == 1)
    # 도메인 밖 표본은 가중치 0 처리 (행 삭제 없이 점추정 기여 0), 도메인 안은 가중치 정규화
    w_all = np.where(d, data[w_col], 0.0)
    w_all = np.where(d, w_all / w_all[d].mean(), 0.0)

    y_ser = data['obesity'].fillna(0).values

    # 설계행렬 구성 (도메인 밖 결측은 임시 값으로 채우되 w_all=0이므로 추정에 전혀 반영되지 않음)
    tmp = data[['bf_skip', 'sex_bin', 'GRADE', ses_col]].copy()
    for v in ['bf_skip', 'sex_bin', 'GRADE', ses_col]:
        tmp[v] = tmp[v].fillna(tmp.loc[d, v].mode()[0])

    if use_covariates:
        X_df = pd.get_dummies(
            tmp[['bf_skip', 'sex_bin', 'GRADE', ses_col]],
            columns=['sex_bin', 'GRADE', ses_col],
            drop_first=True,
            dtype=float
        )
    else:
        X_df = tmp[['bf_skip']].astype(float)

    X_df.insert(0, 'Intercept', 1.0)
    X = X_df.values
    col_names = list(X_df.columns)

    # 1) 가중 최대우도(WML) 추정치 산출 (수치적 안정성을 위해 Pseudo-inverse 또는 Ridge 규제 도입)
    beta = np.zeros(X.shape[1])
    for _ in range(30):
        eta = np.clip(X @ beta, -30, 30)
        mu = 1.0 / (1.0 + np.exp(-eta))
        V = mu * (1.0 - mu)
        score = X.T @ (w_all * (y_ser - mu))
        hess = X.T @ (X * (w_all * V)[:, None])

        # Singular Matrix 방지를 위해 대각선에 미세한 값 추가 (Ridge Regularization 효과)
        hess_reg = hess + 1e-6 * np.eye(hess.shape[0])
        try:
            step = la.solve(hess_reg, score, assume_a='pos')
        except la.LinAlgError:
            # solve 실패 시 의사역행렬 사용
            step = la.pinv(hess_reg) @ score

        beta += step
        if np.max(np.abs(step)) < 1e-10:
            break

    # 2) 설계 기반 샌드위치 분산-공분산 행렬 (층 STRATA, 집락 CLUSTER 반영)
    eta = np.clip(X @ beta, -30, 30)
    mu = 1.0 / (1.0 + np.exp(-eta))
    V = mu * (1.0 - mu)

    hess_final = X.T @ (X * (w_all * V)[:, None]) + 1e-6 * np.eye(X.shape[1])
    hess_inv = la.pinv(hess_final)

    # 개인별 점수 잔차 (도메인 밖은 w_all=0이므로 자동 0)
    U = X * (w_all * (y_ser - mu))[:, None]
    u_df = pd.DataFrame(U, columns=col_names)
    u_df['h'] = data[h_col].values
    u_df['c'] = data[c_col].values

    # 집락(CLUSTER)별 합산 후 층(STRATA) 내 분산 계산
    psu_sum = u_df.groupby(['h', 'c'])[col_names].sum().reset_index()
    G_meat = np.zeros((len(col_names), len(col_names)))
    for _, g in psu_sum.groupby('h'):
        nh = len(g)
        if nh > 1:
            g_mat = g[col_names].values
            g_diff = g_mat - g_mat.mean(axis=0, keepdims=True)
            G_meat += (nh / (nh - 1)) * (g_diff.T @ g_diff)

    cov_beta = hess_inv @ G_meat @ hess_inv
    se_beta = np.sqrt(np.diag(cov_beta))

    idx = col_names.index('bf_skip')
    b_val = beta[idx]
    se_val = se_beta[idx]
    z_val = b_val / se_val
    p_val = 2 * (1 - norm.cdf(abs(z_val)))

    return b_val, se_val, np.exp(b_val), np.exp(b_val - 1.96 * se_val), np.exp(b_val + 1.96 * se_val), p_val

# 모형 1 (노출만) & 모형 2 (보정집합 포함) - 동일 도메인(in_complete == 1)에서 적합
b1, se1, or1, l1, h1, p1 = fit_svy_domain_logit(False, df)
b2, se2, or2, l2, h2, p2 = fit_svy_domain_logit(True, df)

n_tot = int(df['in_complete'].sum())
n_skip = int(((df['in_complete'] == 1) & (df['bf_skip'] == 1)).sum())
n_nonskip = int(((df['in_complete'] == 1) & (df['bf_skip'] == 0)).sum())

table2_df = pd.DataFrame([
    {
        '노출 구분': '아침식사 비결식군 (주 0–4회, Ref)',
        '분석 인원 (n)': f"{n_nonskip:,}",
        'Model 1: OR (95% CI)': '1.00 (Reference)',
        'Model 1 p-value': '-',
        'Model 2: OR (95% CI)': '1.00 (Reference)',
        'Model 2 p-value': '-'
    },
    {
        '노출 구분': '아침식사 결식군 (주 5회 이상)',
        '분석 인원 (n)': f"{n_skip:,}",
        'Model 1: OR (95% CI)': f"{or1:.2f} ({l1:.2f}–{h1:.2f})",
        'Model 1 p-value': f"{p1:.4f}",
        'Model 2: OR (95% CI)': f"{or2:.2f} ({l2:.2f}–{h2:.2f})",
        'Model 2 p-value': f"{p2:.4f}"
    }
])

print(f"=== [C-2] Table 2: 설계 기반 도메인 로지스틱 회귀분석 결과 (전체 {len(df):,}명 유지, 분석 도메인 N = {n_tot:,}명) ===")
print(f"Model 1 (노출만): beta = {b1:.4f}, SE = {se1:.4f}, OR = {or1:.2f} (95% CI: {l1:.2f}–{h1:.2f}), p = {p1:.4f}")
print(f"Model 2 (보정집합): beta = {b2:.4f}, SE = {se2:.4f}, OR = {or2:.2f} (95% CI: {l2:.2f}–{h2:.2f}), p = {p2:.4f}")
display(table2_df)

=== [C-2] Table 2: 설계 기반 도메인 로지스틱 회귀분석 결과 (전체 54,170명 유지, 분석 도메인 N = 25,070명) ===
Model 1 (노출만): beta = 0.0687, SE = 0.0359, OR = 1.07 (95% CI: 1.00–1.15), p = 0.0557
Model 2 (보정집합): beta = 0.0868, SE = 0.0378, OR = 1.09 (95% CI: 1.01–1.17), p = 0.0219


/tmp/ipykernel_2286/942233492.py:84: RuntimeWarning: invalid value encountered in sqrt
  se_beta = np.sqrt(np.diag(cov_beta))


,노출 구분,분석 인원 (n),Model 1: OR (95% CI),Model 1 p-value,Model 2: OR (95% CI),Model 2 p-value
0,"아침식사 비결식군 (주 0–4회, Ref)","13,695",1.00 (Reference),-,1.00 (Reference),-
1,아침식사 결식군 (주 5회 이상),"11,375",1.07 (1.00–1.15),0.0557,1.09 (1.01–1.17),0.0219
